# UNF Roster Turnover Analysis — 2026 → 2027

**Question:** What does the 2027 roster need to produce, statistically, to (1) get back to UNF's own 2026 output, and (2) reach the average of last year's 64-team NCAA Tournament field?

**Method:** Split the 2026 roster into departing vs. returning players using the `2027 Roster` flag, quantify what departing players contributed, then back-solve the rate (AVG/OBP/ERA) incoming players need to produce — assuming they absorb the same playing time (AB / PA / IP) the departing players had.

## Setup

In [1]:
import pandas as pd

DATA_DIR = "/Users/richardnaumann/Documents/UNF 2026 Season"
BENCH_PATH = "/Users/richardnaumann/Documents/UNF-Baseball-Analytics-2026/data/team_benchmarks_clean.csv"

batting = pd.read_csv(f"{DATA_DIR}/UNF_2026_Season_Stats_Batting.csv")
pitching = pd.read_csv(f"{DATA_DIR}/UNF_2026_Season_Stats_Pitching.csv")
benchmarks = pd.read_csv(BENCH_PATH)

## 1. Split the Roster by Return Status

In [2]:
departing_bat = batting[batting["2027 Roster"] == 0]
returning_bat = batting[batting["2027 Roster"] == 1]

departing_pit = pitching[pitching["2027 Roster"] == 0]
returning_pit = pitching[pitching["2027 Roster"] == 1]

bat_counting_columns = ["AB", "H", "2B", "3B", "HR", "RBI", "TB", "BB", "SO", "SF", "HBP"]
pit_counting_columns = ["APP", "GS", "SV", "IP", "H", "R", "ER", "BB", "SO"]

## 2. What's Leaving: Departing Player Production

In [3]:
departing_bat_totals = departing_bat[bat_counting_columns].sum()
team_bat_totals = batting[bat_counting_columns].sum()

departing_pit_totals = departing_pit[pit_counting_columns].sum()
team_pit_totals = pitching[pit_counting_columns].sum()

pct_leaving_bat = departing_bat_totals / team_bat_totals
pct_leaving_pit = departing_pit_totals / team_pit_totals

print("Departing batting totals:\n", departing_bat_totals)
print("\n% of team production leaving (batting):\n", round(pct_leaving_bat, 3))

print("\nDeparting pitching totals:\n", departing_pit_totals)
print("\n% of team production leaving (pitching):\n", round(pct_leaving_pit, 3))

Departing batting totals:
 AB     620
H      154
2B      34
3B       4
HR      26
RBI    117
TB     274
BB      97
SO     158
SF      12
HBP     17
dtype: int64

% of team production leaving (batting):
 AB     0.349
H      0.332
2B     0.337
3B     0.800
HR     0.510
RBI    0.394
TB     0.376
BB     0.372
SO     0.343
SF     0.480
HBP    0.298
dtype: float64

Departing pitching totals:
 APP    125.0
GS      41.0
SV       3.0
IP     320.9
H      299.0
R      187.0
ER     162.0
BB     150.0
SO     311.0
dtype: float64

% of team production leaving (pitching):
 APP    0.592
GS     0.745
SV     0.300
IP     0.692
H      0.687
R      0.665
ER     0.675
BB     0.688
SO     0.673
dtype: float64


**Reading this:** batting losses are moderate (~35% of AB/H, though 51% of HR power). Pitching losses are severe — 69% of innings, 75% of starts, 67% of strikeouts. This is the core finding: the offense needs replacement, the pitching staff needs a near-total rebuild.

## 3. Returning-Only Rates (Current Floor)

In [4]:
returning_bat_totals = returning_bat[bat_counting_columns].sum()
returning_pit_totals = returning_pit[pit_counting_columns].sum()

returning_avg = returning_bat_totals["H"] / returning_bat_totals["AB"]
returning_obp = (
    (returning_bat_totals["H"] + returning_bat_totals["BB"] + returning_bat_totals["HBP"])
    / (returning_bat_totals["AB"] + returning_bat_totals["BB"] + returning_bat_totals["HBP"] + returning_bat_totals["SF"])
)
returning_era = (returning_pit_totals["ER"] * 9) / returning_pit_totals["IP"]

print("Returning AVG:", round(returning_avg, 3))
print("Returning OBP:", round(returning_obp, 3))
print("Returning ERA:", round(returning_era, 2))

Returning AVG: 0.268
Returning OBP: 0.374
Returning ERA: 4.92


## 4. 2026 Team Rates (Target: Match Last Year)

In [5]:
team_avg = team_bat_totals["H"] / team_bat_totals["AB"]
team_obp = (
    (team_bat_totals["H"] + team_bat_totals["BB"] + team_bat_totals["HBP"])
    / (team_bat_totals["AB"] + team_bat_totals["BB"] + team_bat_totals["HBP"] + team_bat_totals["SF"])
)
team_era = (team_pit_totals["ER"] * 9) / team_pit_totals["IP"]

print("Team AVG:", round(team_avg, 3), " Team OBP:", round(team_obp, 3), " Team ERA:", round(team_era, 2))

Team AVG: 0.261  Team OBP: 0.369  Team ERA: 4.66


## 5. Tournament Field Rates (Target: Match the Field)

In [6]:
field = benchmarks[benchmarks["team"] != "North Florida"]

field_obp = field["OBP"].mean()
field_slg = field["SLG"].mean()
field_era = field["ERA"].mean()

print("Field OBP:", round(field_obp, 3), " Field SLG:", round(field_slg, 3), " Field ERA:", round(field_era, 2))

Field OBP: 0.393  Field SLG: 0.47  Field ERA: 5.1


## 6. Back-Solving What Incoming Players Need

**Assumption:** incoming players absorb the same playing time (AB / PA-equivalent for batting, IP for pitching) that departing players had. Given that volume, we solve for the rate (AVG, OBP, ERA) incoming players need so the combined 2027 team hits each target.

General form: `incoming_stat = target * (returning_volume + incoming_volume) - returning_stat`

In [7]:
incoming_ab = departing_bat_totals["AB"]
incoming_ip = departing_pit_totals["IP"]

incoming_pa = (
    departing_bat_totals["AB"] + departing_bat_totals["BB"]
    + departing_bat_totals["HBP"] + departing_bat_totals["SF"]
)
returning_pa = (
    returning_bat_totals["AB"] + returning_bat_totals["BB"]
    + returning_bat_totals["HBP"] + returning_bat_totals["SF"]
)
returning_ob = returning_bat_totals["H"] + returning_bat_totals["BB"] + returning_bat_totals["HBP"]

# --- AVG needed (vs. 2026 team) ---
incoming_h_needed = team_avg * (returning_bat_totals["AB"] + incoming_ab) - returning_bat_totals["H"]
incoming_avg_needed = incoming_h_needed / incoming_ab

# --- OBP needed (vs. 2026 team, vs. tournament field) ---
incoming_obp_needed = (team_obp * (returning_pa + incoming_pa) - returning_ob) / incoming_pa
incoming_obp_needed_field = (field_obp * (returning_pa + incoming_pa) - returning_ob) / incoming_pa

# --- ERA needed (vs. 2026 team, vs. tournament field) ---
incoming_era_needed = ((team_era * (returning_pit_totals["IP"] + incoming_ip)) / 9 - returning_pit_totals["ER"]) * 9 / incoming_ip
incoming_era_needed_field = ((field_era * (returning_pit_totals["IP"] + incoming_ip)) / 9 - returning_pit_totals["ER"]) * 9 / incoming_ip

print("Incoming AVG needed (match 2026 team):", round(incoming_avg_needed, 3))
print("Incoming OBP needed (match 2026 team):", round(incoming_obp_needed, 3))
print("Incoming OBP needed (match tournament field):", round(incoming_obp_needed_field, 3))
print("Incoming ERA needed (match 2026 team):", round(incoming_era_needed, 2))
print("Incoming ERA needed (match tournament field):", round(incoming_era_needed_field, 2))

Incoming AVG needed (match 2026 team): 0.248
Incoming OBP needed (match 2026 team): 0.359
Incoming OBP needed (match tournament field): 0.428
Incoming ERA needed (match 2026 team): 4.54
Incoming ERA needed (match tournament field): 5.18


## 7. Summary

In [8]:
summary = pd.DataFrame({
    "Returning-only": [returning_obp, returning_era],
    "2026 UNF Team": [team_obp, team_era],
    "Incoming needed (match team)": [incoming_obp_needed, incoming_era_needed],
    "Tournament field avg": [field_obp, field_era],
    "Incoming needed (match field)": [incoming_obp_needed_field, incoming_era_needed_field],
}, index=["OBP", "ERA"])

summary.round(3)

,Returning-only,2026 UNF Team,Incoming needed (match team),Tournament field avg,Incoming needed (match field)
OBP,0.374,0.369,0.359,0.393,0.428
ERA,4.919,4.659,4.543,5.102,5.183


**Conclusion:** UNF's two units face opposite challenges heading into 2027.

- **Pitching** was already above tournament-average (~68th percentile in 2026). Defending that level (ERA ≈ 4.54) is a *harder* bar than the tournament average (≈ 5.18) — and 69% of 2026 innings are departing, making this the primary roster risk.
- **Batting** was below tournament-average. Matching UNF's own modest 2026 OBP (≈ .359) is achievable, but climbing to tournament-caliber OBP (≈ .428) requires structural improvement, not just replacement — a gap that existed before any player left.

## 8. Clustering: Which Player *Types* Are Leaving?

The back-solve above tells us how much production is leaving in aggregate. It doesn't tell us whether the losses are spread evenly across player types, or concentrated in a specific kind of player. K-means clustering groups players by statistical profile (not by roster status), so we can check departure rate *within* each type.

Rate stats are used instead of counting stats, since counting stats mostly reflect playing time, not skill profile. Features are standardized before clustering because K-means measures distance between players, and stats on different scales (e.g. AVG's ~0-0.4 range vs. SO's 0-80 range) would otherwise distort that distance.

### 8a. Batting Clusters

In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Minimum playing time filter to avoid small-sample noise
bat_qualified = batting[batting["AB"] >= 20].copy()

# Per-player rate stats (skill profile, not volume)
bat_qualified["AVG"] = bat_qualified["H"] / bat_qualified["AB"]
bat_qualified["ISO"] = (bat_qualified["TB"] - bat_qualified["H"]) / bat_qualified["AB"]
bat_qualified["BB_rate"] = bat_qualified["BB"] / bat_qualified["AB"]
bat_qualified["K_rate"] = bat_qualified["SO"] / bat_qualified["AB"]

bat_features = bat_qualified[["AVG", "ISO", "BB_rate", "K_rate"]]
bat_features_scaled = StandardScaler().fit_transform(bat_features)

bat_kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
bat_qualified["cluster"] = bat_kmeans.fit_predict(bat_features_scaled)

bat_departure_by_cluster = 1 - bat_qualified.groupby("cluster")["2027 Roster"].mean()
bat_cluster_profiles = bat_qualified.groupby("cluster")[["AVG", "ISO", "BB_rate", "K_rate"]].mean()
bat_cluster_counts = bat_qualified["cluster"].value_counts()

print("% departing by cluster:\n", round(bat_departure_by_cluster, 3))
print("\nCluster profiles:\n", round(bat_cluster_profiles, 3))
print("\nPlayers per cluster:\n", bat_cluster_counts)


% departing by cluster:
 cluster
0    0.250
1    0.286
2    0.750
Name: 2027 Roster, dtype: float64

Cluster profiles:
            AVG    ISO  BB_rate  K_rate
cluster                               
0        0.287  0.096    0.159   0.167
1        0.163  0.105    0.094   0.366
2        0.288  0.232    0.172   0.275

Players per cluster:
 1    7
2    4
0    4
Name: cluster, dtype: int64


**Reading this:** Cluster 2 combines the highest AVG, by far the highest ISO (~2.5x the other clusters), and the best walk rate — a complete, middle-of-the-order hitter profile. It is also departing at **75%**, well above the other two, more flawed profiles (contact-only, and high-strikeout bench bats), both departing at ~25-29%. The roster isn't losing offense evenly — it's disproportionately losing its best hitter type.

In [11]:
bat_qualified[bat_qualified["cluster"] == 0][["Player", "AVG", "ISO", "2027 Roster"]]

,Player,AVG,ISO,2027 Roster
1,"Gomez, Gialdri",0.316129,0.161290,1
2,"Benjamin, Sean",0.311765,0.094118,1
3,"Collins, Mitchell",0.308989,0.061798,1
7,"Ordonez, Santiago",0.211382,0.065041,0


In [12]:
bat_qualified[bat_qualified["cluster"] == 1][["Player", "AVG", "ISO", "2027 Roster"]]

,Player,AVG,ISO,2027 Roster
6,"Toberman, Jackson",0.231579,0.147368,1
8,"Buchanan, Drew",0.172727,0.109091,1
12,"Schrafft, Jackson",0.217391,0.144928,1
13,"Alford, Seth",0.201923,0.153846,1
14,"Rurey, Beau",0.130435,0.043478,1
15,"Romeo, Dario",0.100000,0.050000,0
16,"Acosta, Jonathan",0.085714,0.085714,0


In [13]:
bat_qualified[bat_qualified["cluster"] == 2][["Player", "AVG", "ISO", "2027 Roster"]]

,Player,AVG,ISO,2027 Roster
0,"Hosey, Boone",0.320755,0.157233,1
4,"White, Carter",0.301676,0.279330,0
5,"Farner, Mathew",0.286432,0.246231,0
11,"Gerteisen, Tyler",0.243243,0.243243,0


### 8b. Pitching Clusters

In [10]:
# Minimum playing time filter
pit_qualified = pitching[pitching["IP"] >= 10].copy()

# Per-9-innings rate stats
pit_qualified["K/9"] = (pit_qualified["SO"] / pit_qualified["IP"]) * 9
pit_qualified["BB/9"] = (pit_qualified["BB"] / pit_qualified["IP"]) * 9
pit_qualified["H/9"] = (pit_qualified["H"] / pit_qualified["IP"]) * 9

pit_features = pit_qualified[["K/9", "BB/9", "H/9"]]
pit_features_scaled = StandardScaler().fit_transform(pit_features)

pit_kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
pit_qualified["cluster"] = pit_kmeans.fit_predict(pit_features_scaled)

pit_departure_by_cluster = 1 - pit_qualified.groupby("cluster")["2027 Roster"].mean()
pit_cluster_profiles = pit_qualified.groupby("cluster")[["K/9", "BB/9", "H/9"]].mean()
pit_cluster_counts = pit_qualified["cluster"].value_counts()

print("% departing by cluster:\n", round(pit_departure_by_cluster, 3))
print("\nCluster profiles:\n", round(pit_cluster_profiles, 3))
print("\nPlayers per cluster:\n", pit_cluster_counts)


% departing by cluster:
 cluster
0    0.667
1    0.500
2    0.500
Name: 2027 Roster, dtype: float64

Cluster profiles:
             K/9   BB/9     H/9
cluster                       
0         9.996  3.643   6.791
1         7.601  3.243  11.278
2        11.121  9.631   8.315

Players per cluster:
 0    6
1    4
2    2
Name: cluster, dtype: int64


In [14]:
pit_qualified[pit_qualified["cluster"] == 0][["Player", "K/9", "BB/9", "H/9", "2027 Roster"]]

,Player,K/9,BB/9,H/9,2027 Roster
0,"Stone, Dakota",11.620253,4.443038,6.835443,0
1,"Adams, Brandon",13.114286,3.600000,5.914286,1
2,"Nikolis, Trevor",6.577540,4.331551,6.096257,0
3,"Etwaru, Kai",10.431267,3.396226,7.520216,0
6,"Costa, John",8.337292,2.992874,7.268409,0
7,"Diggs, Devin",9.896907,3.092784,7.113402,1


In [15]:
pit_qualified[pit_qualified["cluster"] == 1][["Player", "K/9", "BB/9", "H/9", "2027 Roster"]]

,Player,K/9,BB/9,H/9,2027 Roster
5,"Furey, Ryan",5.563636,2.290909,11.454545,0
8,"Kozera, Tyler",8.640000,4.320000,12.240000,1
9,"Stewart, Jeremiah",6.725979,3.202847,10.889680,1
10,"Hendry, Clay",9.473684,3.157895,10.526316,0


In [16]:
pit_qualified[pit_qualified["cluster"] == 2][["Player", "K/9", "BB/9", "H/9", "2027 Roster"]]

,Player,K/9,BB/9,H/9,2027 Roster
11,"Dimino, Joseph",10.992366,9.618321,6.183206,1
13,"Starling, Zane",11.250000,9.642857,10.446429,0


**Reading this:** the cluster with the highest strikeouts (~10 K/9), lowest walks, and fewest hits allowed — the best all-around pitcher profile, 6 pitchers in this sample — departs at **66.7%**, versus 50% for the other two, flawed-but-usable profiles (a contact/gets-hit arm, and a high-strikeout/high-walk wild arm). The same pattern as batting: the best statistical type is leaving at a higher rate than the rest of the staff.

## 9. Combined Finding

Across both position groups, the roster isn't just losing volume — it's disproportionately losing its **best statistical profile**: power-and-discipline hitters on offense, and strikeout-and-control arms on the mound. The back-solve in Section 6 quantifies how much production needs replacing in aggregate; this clustering result shows that loss is concentrated in the type of player hardest to simply replace with volume alone.